In [1]:
# importing required libraries

import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats

import pickle

In [2]:
df = pd.read_csv("AQI.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29531 entries, 0 to 29530
Data columns (total 16 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   City        29531 non-null  object 
 1   Date        29531 non-null  object 
 2   PM2.5       24933 non-null  float64
 3   PM10        18391 non-null  float64
 4   NO          25949 non-null  float64
 5   NO2         25946 non-null  float64
 6   NOx         25346 non-null  float64
 7   NH3         19203 non-null  float64
 8   CO          27472 non-null  float64
 9   SO2         25677 non-null  float64
 10  O3          25509 non-null  float64
 11  Benzene     23908 non-null  float64
 12  Toluene     21490 non-null  float64
 13  Xylene      11422 non-null  float64
 14  AQI         24850 non-null  float64
 15  AQI_Bucket  24850 non-null  object 
dtypes: float64(13), object(3)
memory usage: 3.6+ MB


In [3]:
df.isnull().sum()/len(df)*100 #verify percentage of missing values

,0
City,0.000000
Date,0.000000
PM2.5,15.570079
PM10,37.723071
NO,12.129626
NO2,12.139785
NOx,14.171549
NH3,34.973418
CO,6.972334
SO2,13.050692


# Observation



*   PM10, NH3, Touloune, Xylene columns have more missing values. These columns must be dropped
*   The data column is redundant in our case and is not required. This too will be dropped.
* AQI bucket is a different version of the output variable. This, again will be dropped, as we are trying to target regression.



In [4]:
df["AQI_Bucket"].value_counts()

,count
AQI_Bucket,
Moderate,8829
Satisfactory,8224
Poor,2781
Very Poor,2337
Good,1341
Severe,1338


In [5]:
df=df.drop(columns=['AQI_Bucket', 'Date', 'PM10', 'NH3','Toluene','Xylene'])
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29531 entries, 0 to 29530
Data columns (total 10 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   City     29531 non-null  object 
 1   PM2.5    24933 non-null  float64
 2   NO       25949 non-null  float64
 3   NO2      25946 non-null  float64
 4   NOx      25346 non-null  float64
 5   CO       27472 non-null  float64
 6   SO2      25677 non-null  float64
 7   O3       25509 non-null  float64
 8   Benzene  23908 non-null  float64
 9   AQI      24850 non-null  float64
dtypes: float64(9), object(1)
memory usage: 2.3+ MB


In [6]:
df.head()

,City,PM2.5,NO,NO2,NOx,CO,SO2,O3,Benzene,AQI
0,Ahmedabad,NaN,0.92,18.22,17.15,0.92,27.64,133.36,0.00,NaN
1,Ahmedabad,NaN,0.97,15.69,16.46,0.97,24.55,34.06,3.68,NaN
2,Ahmedabad,NaN,17.40,19.30,29.70,17.40,29.07,30.70,6.80,NaN
3,Ahmedabad,NaN,1.70,18.48,17.97,1.70,18.59,36.08,4.43,NaN
4,Ahmedabad,NaN,22.10,21.42,37.76,22.10,39.33,39.31,7.01,NaN


In [7]:
df = df.dropna(subset=["AQI"])



In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 24850 entries, 28 to 29530
Data columns (total 10 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   City     24850 non-null  object 
 1   PM2.5    24172 non-null  float64
 2   NO       24463 non-null  float64
 3   NO2      24459 non-null  float64
 4   NOx      22993 non-null  float64
 5   CO       24405 non-null  float64
 6   SO2      24245 non-null  float64
 7   O3       24043 non-null  float64
 8   Benzene  21315 non-null  float64
 9   AQI      24850 non-null  float64
dtypes: float64(9), object(1)
memory usage: 2.1+ MB


In [9]:
df.head() # checking if the above step worked

,City,PM2.5,NO,NO2,NOx,CO,SO2,O3,Benzene,AQI
28,Ahmedabad,83.13,6.93,28.71,33.72,6.93,49.52,59.76,0.02,209.0
29,Ahmedabad,79.84,13.85,28.68,41.08,13.85,48.49,97.07,0.04,328.0
30,Ahmedabad,94.52,24.39,32.66,52.61,24.39,67.39,111.33,0.24,514.0
31,Ahmedabad,135.99,43.48,42.08,84.57,43.48,75.23,102.70,0.40,782.0
32,Ahmedabad,178.33,54.56,35.31,72.80,54.56,55.04,107.38,0.46,914.0


In [10]:
# We now deal with duplicates

df=df.drop_duplicates()

In [11]:
df.shape # This shows us that the step worked.

(24850, 10)

In [12]:
# AQI is the target variable
# All the remaining columns are predictors/input variable

In [13]:
y = df["AQI"]
print(y.shape)

(24850,)


In [14]:
X = df.drop(columns="AQI")
X.shape

(24850, 9)

In [15]:
# Splitting data into training and testing

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size = 0.2, random_state=42)
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

(19880, 9)
(19880,)
(4970, 9)
(4970,)


In [16]:
# next we check what are the unique values

transformer = ColumnTransformer([
    ("city", OneHotEncoder(handle_unknown="ignore", sparse_output=False), [0]),
    ("pollutants", make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler()
    ), [1, 2, 3, 4, 5, 6, 7, 8])
])

# We fit transform on trained data
X_train_trans = transformer.fit_transform(X_train)

# Let us transform Test data
X_test_trans = transformer.transform(X_test)

In [17]:

pd.DataFrame(X_train_trans).head()

,0,1,2,3,4,5,6,7,8,9,...,24,25,26,27,28,29,30,31,32,33
0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,-0.733075,-0.237439,-0.210274,-0.302542,-0.096977,-0.141930,-0.271675,-0.211773
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,-0.821936,-0.742535,-0.832184,-0.888258,-0.287305,-0.364996,0.336604,-0.015490
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.070256,-0.605878,-0.555961,-0.912747,0.016074,-0.391579,0.426580,-0.152399
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,-0.552914,-0.740302,-0.961306,-0.941932,-0.235788,-0.682836,0.877400,-0.128650
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.699436,1.195229,1.095654,1.690438,-0.128460,0.361991,0.801482,0.004767


# Building the KNN Model

In [18]:
from sklearn.neighbors import KNeighborsRegressor
knn=KNeighborsRegressor()
# fit the data
knn.fit(X_train_trans,y_train)

KNeighborsRegressor()

In [19]:
y_pred=knn.predict(X_test_trans)
print(y_pred.shape)
print(y_test.shape)

(4970,)
(4970,)


# Time for model validation

In [20]:
'''This will be achieved by using the following metrics:

- Mean Absolute Error (MAE)

- Mean Squared Error (MSE)

- Root Mean Squared Error (RMSE)

- r2 score (Coefficient of deternination)

- adjusted r2 score '''

'This will be achieved by using the following metrics:\n\n- Mean Absolute Error (MAE)\n\n- Mean Squared Error (MSE)\n\n- Root Mean Squared Error (RMSE)\n\n- r2 score (Coefficient of deternination)\n\n- adjusted r2 score '

In [21]:
from sklearn.metrics import mean_absolute_error

MAE_score = mean_absolute_error(y_test, y_pred)
print("Mean Absolute Error/MAE of knn model is: ", MAE_score)

Mean Absolute Error/MAE of knn model is:  25.223702213279676


In [22]:
from sklearn.metrics import mean_squared_error

MSE_score = mean_squared_error(y_test, y_pred)
print("Mean Squared Error/MSE of knn model is: ", MSE_score)

Mean Squared Error/MSE of knn model is:  2497.0421327967806


In [23]:
from sklearn.metrics import root_mean_squared_error

RMSE_score = root_mean_squared_error(y_test, y_pred)
print("Root Mean Squared Error/RMSE of knn model is: ", RMSE_score)

Root Mean Absolute Error/RMSE of knn model is:  49.970412573809924


In [24]:
from sklearn.metrics import r2_score

r2score = r2_score(y_test, y_pred)
print("r2 score of knn model is: ", r2score)

r2 score of knn model is:  0.8636317379151981


In [25]:
# adjusted r squared:

r2score = r2_score(y_test, y_pred)

n = len(y_test)                 # number of test rows
p = X_test_trans.shape[1]      # number of input columns after transformation

adjusted_r2 = 1 - (1 - r2score) * (n - 1) / (n - p - 1)

print("r2score: ", r2score)
print("Adjusted r2 score:", adjusted_r2)

r2score:  0.8636317379151981
Adjusted r2 score: 0.8626922200001255


In [26]:
# save the model to disk

with open("finalized_model.sav", "wb") as file:
    pickle.dump({"model": knn, "transformer": transformer}, file)